# Introduction to Quantum Computing with Quantumium

This notebook covers the fundamentals of quantum computing using the Quantumium SDK:

1. Qubits and quantum states
2. Quantum gates
3. Building circuits
4. Measurement and probabilities
5. Multi-qubit entanglement
6. Noisy simulation

In [ ]:
import numpy as np
from quantumium.core.qubit import Qubit
from quantumium.core.gate import H, X, Y, Z, CNOT, S, T, Rx, Ry, Rz
from quantumium.core.circuit import QuantumCircuit
from quantumium.core.noise import NoiseModel, depolarizing, amplitude_damping

## 1. Qubits — The Building Blocks

A qubit is the quantum analogue of a classical bit. While a classical bit is either 0 or 1,
a qubit can be in a **superposition** of both states:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

where $|\alpha|^2 + |\beta|^2 = 1$.

In [ ]:
# Create a qubit in the |0> state (default)
q = Qubit()
print("State |0>:", q.state)
p0, p1 = q.probabilities
print(f"Probabilities: P(0) = {p0}, P(1) = {p1}")

In [ ]:
# Create a qubit in the |1> state
q1 = Qubit(0, 1)
print("State |1>:", q1.state)
p0, p1 = q1.probabilities
print(f"Probabilities: P(0) = {p0}, P(1) = {p1}")

In [ ]:
# Create a custom superposition: (|0> + i|1>) / sqrt(2)
q_custom = Qubit(1/np.sqrt(2), 1j/np.sqrt(2))
print("Custom state:", q_custom.state)
p0, p1 = q_custom.probabilities
print(f"Probabilities: P(0) = {p0:.4f}, P(1) = {p1:.4f}")

## 2. Quantum Gates

Quantum gates are unitary operations that transform qubit states. Common single-qubit gates:

| Gate | Effect |
|------|--------|
| **X** (NOT) | Flips \|0⟩ ↔ \|1⟩ |
| **H** (Hadamard) | Creates equal superposition |
| **Z** | Adds phase of -1 to \|1⟩ |
| **S, T** | Phase gates (π/2 and π/4) |
| **Rx, Ry, Rz** | Rotation gates |

In [ ]:
# Hadamard gate: |0> -> (|0> + |1>) / sqrt(2)
qc = QuantumCircuit(1)
qc.apply(H, 0)
sv = qc.statevector()
print("After H|0>:", np.round(sv, 4))
print("This is an equal superposition — 50/50 chance of measuring 0 or 1")

In [ ]:
# X gate (quantum NOT): |0> -> |1>
qc = QuantumCircuit(1)
qc.apply(X, 0)
sv = qc.statevector()
print("After X|0>:", np.round(sv, 4))

In [ ]:
# Rotation gates take an angle parameter
qc = QuantumCircuit(1)
qc.apply(Rx(np.pi / 4), 0)  # Rotate by pi/4 around X axis
sv = qc.statevector()
probs = np.abs(sv) ** 2
print(f"After Rx(π/4)|0>: {np.round(sv, 4)}")
print(f"Probabilities: P(0) = {probs[0]:.4f}, P(1) = {probs[1]:.4f}")

## 3. Building Quantum Circuits

A quantum circuit combines multiple gates applied to multiple qubits in sequence.

In [ ]:
# Build a 2-qubit circuit
qc = QuantumCircuit(2)

# Apply gates
qc.apply(H, 0)       # Hadamard on qubit 0
qc.apply(X, 1)       # NOT on qubit 1
qc.apply(CNOT, 0, 1) # CNOT with control=0, target=1

sv = qc.statevector()
print("Statevector:", np.round(sv, 4))
print("\nAmplitudes (basis states |q1 q0>):")
for i, amp in enumerate(sv):
    if abs(amp) > 1e-10:
        print(f"  |{i:02b}> : {amp:.4f}  (probability {abs(amp)**2:.4f})")

## 4. Measurement

Measurement collapses a quantum state into a classical outcome. The probability of each
outcome is determined by the amplitudes of the statevector.

In [ ]:
# Create a superposition and measure many times
qc = QuantumCircuit(2)
qc.apply(H, 0)
qc.apply(H, 1)

# This puts both qubits in superposition: each of |00>, |01>, |10>, |11> has 25% probability
counts = qc.measure_all(shots=1000)

print("Measurement results (1000 shots):")
for bitstring, count in sorted(counts.items()):
    print(f"  |{bitstring}> : {count:4d} times ({count/10:.1f}%)")

## 5. Entanglement — Bell States

Entanglement is a uniquely quantum phenomenon where two qubits become correlated in ways
that have no classical analogue. A **Bell state** is the simplest example:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

Measuring one qubit instantly determines the other, regardless of distance.

In [ ]:
# Create a Bell state: |Φ+> = (|00> + |11>) / sqrt(2)
qc = QuantumCircuit(2)
qc.apply(H, 0)
qc.apply(CNOT, 0, 1)

sv = qc.statevector()
print("Bell state |Φ+>:", np.round(sv, 4))

# Measure — should only see |00> and |11>, never |01> or |10>
counts = qc.measure_all(shots=1000)
print("\nMeasurement results:")
for bitstring, count in sorted(counts.items()):
    print(f"  |{bitstring}> : {count:4d} ({count/10:.1f}%)")
print("\nNotice: only |00> and |11> appear — the qubits are perfectly correlated!")

In [ ]:
# GHZ state: 3-qubit entanglement (|000> + |111>) / sqrt(2)
qc = QuantumCircuit(3)
qc.apply(H, 0)
qc.apply(CNOT, 0, 1)
qc.apply(CNOT, 0, 2)

counts = qc.measure_all(shots=1000)
print("GHZ state measurement (3 qubits):")
for bitstring, count in sorted(counts.items()):
    if count > 0:
        print(f"  |{bitstring}> : {count:4d} ({count/10:.1f}%)")
print("\nAll three qubits are correlated — either all 0 or all 1.")

## 6. Noisy Simulation

Real quantum hardware is noisy. Quantumium can simulate common noise channels
to model realistic device behavior.

In [ ]:
# Compare ideal vs noisy Bell state
print("Ideal Bell state:")
qc_ideal = QuantumCircuit(2)
qc_ideal.apply(H, 0)
qc_ideal.apply(CNOT, 0, 1)
counts_ideal = qc_ideal.measure_all(shots=1000)
for bs, c in sorted(counts_ideal.items()):
    print(f"  |{bs}> : {c:4d} ({c/10:.1f}%)")

print("\nNoisy Bell state (5% depolarizing noise):")
noise = NoiseModel(default_single_qubit_noise=depolarizing(0.05),
                   default_two_qubit_noise=depolarizing(0.05))
qc_noisy = QuantumCircuit(2)
qc_noisy.apply(H, 0)
qc_noisy.apply(CNOT, 0, 1)
counts_noisy = qc_noisy.noisy_measure(noise, shots=1000)
for bs, c in sorted(counts_noisy.items()):
    print(f"  |{bs}> : {c:4d} ({c/10:.1f}%)")

print("\nWith noise, |01> and |10> appear — the entanglement is degraded.")

In [ ]:
# Amplitude damping simulates energy loss (T1 decay)
print("Amplitude damping (γ=0.1) on |1> state:")
noise_ad = NoiseModel(default_single_qubit_noise=amplitude_damping(0.1))
qc = QuantumCircuit(1)
qc.apply(X, 0)  # Prepare |1>
counts = qc.noisy_measure(noise_ad, shots=1000)
for bs, c in sorted(counts.items()):
    print(f"  |{bs}> : {c:4d} ({c/10:.1f}%)")
print("Some probability has 'decayed' from |1> to |0>.")

## Summary

In this notebook you learned:

- **Qubits** can exist in superpositions of |0⟩ and |1⟩
- **Quantum gates** (H, X, CNOT, rotations) transform qubit states
- **Circuits** compose gates into quantum programs
- **Measurement** collapses superpositions into classical outcomes
- **Entanglement** creates non-classical correlations between qubits
- **Noise models** simulate realistic quantum hardware

Next: explore [quantum algorithms](quantum-algorithms.ipynb) to see how these primitives solve real problems.